In [1]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import math
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
from PIL import Image

In [4]:
from utils.generic_utils import SQLModule
from utils.data_utils import StockPriceProcess
from utils.constants import CURRENCY_MAPPER, PLOTLY_CURRENCY_NORMALIZER

In [5]:
CODES = {
    'vietnam' : ['ACB'],
    'australia' : ['TPG', 'TNE', 'SGLLV']
}
CODES_TO_COUNTRY = {v : k for k,vv in CODES.items() for v in vv}
FEATURES = ['close', 'sma_close', 'upper_band', 'lower_band']

Z_SCORE = 3
DAYS = 365
PERIOD = 27

In [6]:
# Get 1 year data
end_date = datetime.today().date()
start_date = end_date - timedelta(days = DAYS)
world_engine = SQLModule.get_engine(country = 'world')
# Get all exchange rate
query = f"""
    SELECT
        *
    FROM daily_average_exchange_rate_usd_based
    WHERE
        date >= DATE '{start_date}'
        AND
        date <= DATE '{end_date}'
    ORDER BY date
"""
ex_rate = pd.read_sql_query(query, world_engine)
ex_rate.set_index('date', inplace = True)
# fill nan for each rate
for col in ex_rate.columns:
    # Backfilling the variables
    ex_rate[col] = ex_rate[col].fillna(method = 'bfill').fillna(method = 'ffill')

C:\Users\khoan\AppData\Local\Temp\ipykernel_25936\746384279.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ex_rate[col] = ex_rate[col].fillna(method = 'bfill').fillna(method = 'ffill')


In [7]:
df = None
# Bollinger band analysis
columns = []
for stock_code,country in CODES_TO_COUNTRY.items():
    engine = SQLModule.get_engine(country = country)
    stock_query = f"""
        SELECT
            stock_code,
            date,
            close
        FROM transaction
        WHERE
            stock_code = '{stock_code}'
            AND
            date >= DATE '{start_date}'
            AND
            date <= DATE '{end_date}'
        ORDER BY date
    """
    _df = pd.read_sql_query(stock_query, engine)
    _df.index = _df['date']
    if country != 'united_states':
        _df = _df.join(ex_rate[[CURRENCY_MAPPER[country]]])
        _df['close'] = _df['close'] / _df[CURRENCY_MAPPER[country]]
        _df.drop(columns = [CURRENCY_MAPPER[country]], inplace = True)
    _df = _df.reset_index(drop = True)
    df = pd.concat([df, _df])
    columns += [(stock_code, x) for x in FEATURES]
df = StockPriceProcess.frame_var(df)
df = StockPriceProcess.remove_invalid_data(df, country = country)

columns = pd.MultiIndex.from_tuples(columns, names = ['Stock code', 'Feature'])
result_df = pd.DataFrame(columns = columns)
for country in CODES:
    for stock_code in CODES[country]:
        # Save the data
        result_df[(stock_code, 'close')] = df[stock_code]

        # Compute bollinger band analysis
        result_df[(stock_code, 'sma_close')] = df[stock_code].rolling(PERIOD).mean()
        result_df[(stock_code, 'smsd_close')] = df[stock_code].rolling(PERIOD).std()
        
        result_df[(stock_code, 'upper_band')] = result_df[(stock_code, 'sma_close')] + result_df[(stock_code, 'smsd_close')] * Z_SCORE
        result_df[(stock_code, 'lower_band')] = result_df[(stock_code, 'sma_close')] - result_df[(stock_code, 'smsd_close')] * Z_SCORE

# reduce the column
columns = []
for country in CODES:
    for stock_code in CODES[country]:
        columns += [(stock_code, x) for x in FEATURES]
result_df = result_df[columns]
df = result_df.iloc[PERIOD - 1:,:]
df.iloc[-5:,:]

C:\Users\khoan\OneDrive\Documents\stock_data_scraper\utils\data_utils.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  no_holiday_df[col] = no_holiday_df[col].fillna(method = 'bfill').fillna(method = 'ffill')
C:\Users\khoan\OneDrive\Documents\stock_data_scraper\utils\data_utils.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_holiday_df[col] = no_holiday_df[col].fillna(method = 'bfill').fillna(method = 'ffill')


Stock code       ACB                                       TPG            \
Feature        close sma_close upper_band lower_band     close sma_close   
date                                                                       
2025-02-12  1.004700  0.995741   1.039513   0.951969  2.752357  2.735508   
2025-02-13  1.005871  0.996499   1.040193   0.952804  2.707967  2.734129   
2025-02-14  1.012186  0.997476   1.041588   0.953363  2.773128  2.735462   
2025-02-17  1.016949  0.998717   1.043378   0.954055  2.784152  2.737813   
2025-02-18  1.016581  1.000316   1.043436   0.957196  2.812224  2.742017   

Stock code                              TNE                                   \
Feature    upper_band lower_band      close  sma_close upper_band lower_band   
date                                                                           
2025-02-12   2.879484   2.591532  20.066384  18.915715  21.235423  16.596007   
2025-02-13   2.878840   2.589418  19.979895  18.943991  21.338627  16.549355   
2025-02-14   2.881914   2.589011  20.340482  19.001341  21.525349  16.477333   
2025-02-17   2.886613   2.589013  20.366263  19.057646  21.699146  16.416146   
2025-02-18   2.894869   2.589166  20.487241  19.120665  21.881509  16.359820   

Stock code     SGLLV                                  
Feature        close sma_close upper_band lower_band  
date                                                  
2025-02-12  6.531337  6.503125   7.121781   5.884468  
2025-02-13  6.502891  6.517516   7.090641   5.944392  
2025-02-14  6.500110  6.529878   7.063224   5.996532  
2025-02-17  6.706120  6.543593   7.073138   6.014047  
2025-02-18  6.680622  6.557419   7.072982   6.041856

In [8]:
fig = make_subplots(
    rows = len(CODES_TO_COUNTRY), 
    cols = 1, 
    subplot_titles = [f'[{country}] {stock_code}' for stock_code, country in CODES_TO_COUNTRY.items()],
)
fig.update_annotations(font = dict(size = 30))
for num,(stock_code,country) in enumerate(CODES_TO_COUNTRY.items()):
    # Get data from that stock code
    _df = df[[(stock_code, col) for col in FEATURES]].droplevel(0, axis = 1)
    # merge with ex_rate
    _df = _df.join(ex_rate[[CURRENCY_MAPPER[country]]])
    for col in FEATURES:
        _df[col] = _df[col] * _df[CURRENCY_MAPPER[country]]

    # Plot the upper and lower band
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['lower_band']  / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  
            marker = dict(color = 'red'),
            name = 'Lower band price',
            mode = 'lines',
            showlegend = num == 0
        ),
        row = num + 1, col = 1
    )
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['upper_band'] / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  
            marker = dict(color = 'green'),
            name = 'Upper band price',
            mode = 'lines',
            fill = 'tonexty',
            showlegend = num == 0
        ),
        row = num + 1, col = 1
    )

    # Plot the close price
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['close'] / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  
            marker = dict(color = 'blue'),
            name = 'Close price',
            mode = 'lines',
            showlegend = num == 0
        ),
        row = num + 1, col = 1
    )

    # plot the sma close
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['sma_close'] / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  
            marker = dict(color = 'yellow'),
            name = 'SMA close price',
            mode = 'lines',
            showlegend = num == 0
        ),
        row = num + 1, col = 1
    )

    # Plot the buy sell signals
    for date, row in _df.iterrows():
        close_price = row['close']
        upper_price = row['upper_band']
        lower_price = row['lower_band']
        if close_price < lower_price: #oversold condition
            fig.add_layout_image(
                dict(
                    source = Image.open("img/bull_icon.png"),
                    x = date,  # x location (date)
                    xanchor = "center",
                    yanchor = 'top',
                    y = close_price / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  # y location (price)
                    xref="x",  # Referencing the x-axis
                    yref="y",  # Referencing the y-axis
                    sizex = 5 * 24 * 60 * 60 * 1000,  # Image width
                    sizey = 5,    # Image height
                    sizing="contain",
                    opacity = 1.0,
                    layer="above"
                ),
                row = num + 1, col = 1
            )
        elif close_price > upper_price: #overbought condition
            fig.add_layout_image(
                dict(
                    source = Image.open("img/bear_icon.png"),
                    x = date,  # x location (date)
                    xanchor = "center",
                    yanchor = 'bottom',
                    y = close_price / PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]],  # y location (price)
                    xref="x",
                    yref="y",
                    sizex = 5 * 24 * 60 * 60 * 1000,  # Image width
                    sizey = 5,    # Image height
                    sizing="contain",
                    opacity = 1.0,
                    layer="above"
                ),
                row = num + 1, col = 1
            )

    fig.update_yaxes(
        title = 'Close price',
        tickprefix = f'{CURRENCY_MAPPER[CODES_TO_COUNTRY[stock_code]]} ',
        ticksuffix = f' * {PLOTLY_CURRENCY_NORMALIZER[CURRENCY_MAPPER[country]]}', 
        showgrid = True, 
        gridcolor = 'gray',
        row = num + 1, col = 1
    )
fig.update_xaxes(title = 'Date', showgrid = False)
fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    height = 400 * len(CODES_TO_COUNTRY),
    font = dict(size = 20),
    title = stock_code
)
fig.show()